In [1]:
%pip install transformers datasets tensorflow scikit-learn pandas
%pip install transformers==4.41.2 tokenizers==0.19.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 265.3 kB/s eta 0:00:00 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 325.1 kB/s eta 0:00:001m112.7 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 350.0 kB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 21.4 MB/s eta 0:00:00m eta 0:00:01:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 2.8 MB/s eta 0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 383.5 kB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 16.4 MB/s eta 0:00:00m eta 0:00:0136m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 294.9 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 158.3 kB/s eta 0:00:0031m55.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 363.5 kB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import BertTokenizer

# Load dataset
df = pd.read_csv(r"/home/bhcp0089/Desktop/AiMedicalChatbot_updated/ophthalmology_mixed_1000_dataset.csv")  #ophthalmology_classification_dataset

print(df.columns)

# Convert labels (Medical = 1, Non-Medical = 0)
df['Category'] = df['Category'].map({'Ophthalmology': 1, 'Non-Ophthalmology': 0})

# Train-test split (80-20)
from sklearn.model_selection import train_test_split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["Query"].tolist(), df["Category"].tolist(), test_size=0.2, random_state=42
)

# Load BERT tokenizer
# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased",
    use_auth_token=False   # 🔥 stronger than token=None
)

# Tokenize queries
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)



Index(['Query', 'Category'], dtype='str')


/home/bhcp0089/Desktop/AiMedicalChatbot_updated/myenv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1974: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/bhcp0089/Desktop/AiMedicalChatbot_updated/myenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
# Convert to TensorFlow Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).batch(16)  # Batch size 16

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    test_labels
)).batch(16)  # Batch size 16


E0000 00:00:1776835412.303159    9457 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [11]:
%pip install tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.3 MB/s eta 0:00:000m eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [13]:
from transformers import TFBertForSequenceClassification

# Load BERT with 2 output classes
# model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_auth_token=False
)


/home/bhcp0089/Desktop/AiMedicalChatbot_updated/myenv/lib/python3.12/site-packages/transformers/modeling_tf_utils.py:2696: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
W0000 00:00:1776835626.612637    9457 cpu_allocator_impl.cc:82] Allocation of 93763584 exceeds 10% of free system memory.
W0000 00:00:1776835627.910317    9457 cpu_allocator_impl.cc:82] Allocation of 93763584 exceeds 10% of free system memory.
W0000 00:00:1776835628.165958    9457 cpu_allocator_impl.cc:82] Allocation of 93763584 exceeds 10% of free system memory.
W0000 00:00:1776835632.780385    9457 cpu_allocator_impl.cc:82] Allocation of 93763584 exceeds 10% of free system memory.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['class

In [14]:
# Compile model with optimizer, loss, and metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


In [15]:
# Train the model
model.fit(train_dataset, validation_data=test_dataset, epochs=10)


Epoch 1/10


W0000 00:00:1776835646.903206    9457 cpu_allocator_impl.cc:82] Allocation of 93763584 exceeds 10% of free system memory.


50/50 [==============================] - 327s 6s/step - loss: 0.1959 - accuracy: 0.9650 - val_loss: 0.0169 - val_accuracy: 1.0000
Epoch 2/10
50/50 [==============================] - 254s 5s/step - loss: 0.0094 - accuracy: 1.0000 - val_loss: 0.0033 - val_accuracy: 1.0000
Epoch 3/10
50/50 [==============================] - 256s 5s/step - loss: 0.0031 - accuracy: 1.0000 - val_loss: 0.0016 - val_accuracy: 1.0000
Epoch 4/10
50/50 [==============================] - 248s 5s/step - loss: 0.0017 - accuracy: 1.0000 - val_loss: 9.6080e-04 - val_accuracy: 1.0000
Epoch 5/10
50/50 [==============================] - 412s 8s/step - loss: 0.0011 - accuracy: 1.0000 - val_loss: 6.2779e-04 - val_accuracy: 1.0000
Epoch 6/10
50/50 [==============================] - 370s 7s/step - loss: 7.8563e-04 - accuracy: 1.0000 - val_loss: 4.4923e-04 - val_accuracy: 1.0000
Epoch 7/10
13/50 [======>.......................] - ETA: 4:30 - loss: 6.4535e-04 - accuracy: 1.0000

IOStream.flush timed out


50/50 [==============================] - 382s 8s/step - loss: 5.9171e-04 - accuracy: 1.0000 - val_loss: 3.4313e-04 - val_accuracy: 1.0000
Epoch 8/10
50/50 [==============================] - 258s 5s/step - loss: 4.6297e-04 - accuracy: 1.0000 - val_loss: 2.7129e-04 - val_accuracy: 1.0000
Epoch 9/10
50/50 [==============================] - 243s 5s/step - loss: 3.8568e-04 - accuracy: 1.0000 - val_loss: 2.2309e-04 - val_accuracy: 1.0000
Epoch 10/10
50/50 [==============================] - 266s 5s/step - loss: 3.1855e-04 - accuracy: 1.0000 - val_loss: 1.8816e-04 - val_accuracy: 1.0000


In [16]:
# Evaluate model accuracy on test data
loss, accuracy = model.evaluate(test_dataset)
print(f"Test Accuracy: {accuracy:.4f}")


13/13 [==============================] - 15s 1s/step - loss: 1.8816e-04 - accuracy: 1.0000
Test Accuracy: 1.0000


In [17]:
# Save trained model
model.save_pretrained("bert_medical_classifier_tf")
tokenizer.save_pretrained("bert_medical_classifier_tf")

# Load model later
from transformers import TFBertForSequenceClassification, BertTokenizer
model = TFBertForSequenceClassification.from_pretrained("bert_medical_classifier_tf")
tokenizer = BertTokenizer.from_pretrained("bert_medical_classifier_tf")


Some layers from the model checkpoint at bert_medical_classifier_tf were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at bert_medical_classifier_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


In [18]:
from transformers import TFBertForSequenceClassification, BertTokenizer
import tensorflow as tf

# Load trained model
model = TFBertForSequenceClassification.from_pretrained("bert_medical_classifier_tf")
tokenizer = BertTokenizer.from_pretrained("bert_medical_classifier_tf")


Some layers from the model checkpoint at bert_medical_classifier_tf were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at bert_medical_classifier_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


In [19]:
def classify_query(query):
    # Tokenize input text
    inputs = tokenizer(query, return_tensors="tf", truncation=True, padding=True, max_length=128)

    # Get model predictions
    outputs = model(**inputs)

    # Convert logits to class prediction
    predicted_class = tf.argmax(outputs.logits, axis=1).numpy()[0]

    # Return classification result
    return "Ophthalmology" if predicted_class == 1 else "Non-Ophthalmology"


In [20]:
while True:
    user_query = input("Enter your query (or type 'exit' to stop): ")

    if user_query.lower() == "exit":
        print("Exiting the program. Goodbye!")
        break  # Stop the loop

    prediction = classify_query(user_query)  # Call the classification function
    print(f"input: {user_query}\n")
    print(f"Prediction: {prediction}\n")


input: red eyes

Prediction: Ophthalmology

input: good morning

Prediction: Non-Ophthalmology

input: myopia

Prediction: Ophthalmology

input: glucoma

Prediction: Ophthalmology

input: cricket

Prediction: Non-Ophthalmology

input: fever

Prediction: Ophthalmology

Exiting the program. Goodbye!


In [1]:
!pip install transformers datasets torch scikit-learn pandas


  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.23.4
    Uninstalling huggingface-hub-0.23.4:
      Successfully uninstalled huggingface-hub-0.23.4


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import torch

# Load dataset
df = pd.read_csv(r"/home/bhcp0089/Desktop/AiMedicalChatbot_updated/ophthalmology_mixed_1000_dataset.csv")  #  ophthalmology_classification_dataset

print(df.columns)

# Convert labels (Ophthalmology = 1, Non-Ophthalmology = 0)
df['Category'] = df['Category'].map({'Ophthalmology': 1, 'Non-Ophthalmology': 0})

# Train-test split (80-20)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["Query"].tolist(), df["Category"].tolist(), test_size=0.2, random_state=42
)

# Load BERT tokenizer
# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased",
    use_auth_token=False   # 🔥 stronger than token=None
)

# Tokenize queries
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

# Convert labels to tensors
train_labels = torch.tensor(train_labels)
test_labels = torch.tensor(test_labels)


/home/bhcp0089/Desktop/AiMedicalChatbot_updated/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Index(['Query', 'Category'], dtype='str')


/home/bhcp0089/Desktop/AiMedicalChatbot_updated/myenv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1974: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [3]:
from torch.utils.data import Dataset

class QueryDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# Create dataset objects
train_dataset = QueryDataset(train_encodings, train_labels)
test_dataset = QueryDataset(test_encodings, test_labels)


In [4]:
from transformers import BertForSequenceClassification

# Load BERT with 2 output classes (Ophthalmology & Non-Ophthalmology)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# !pip install "transformers[torch]" -U


In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

/home/bhcp0089/Desktop/AiMedicalChatbot_updated/myenv/lib/python3.12/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [7]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()


  7%|▋         | 10/150 [00:48<10:45,  4.61s/it]

{'loss': 0.7674, 'grad_norm': 8.311559677124023, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.2}


 13%|█▎        | 20/150 [01:32<09:04,  4.19s/it]

{'loss': 0.7382, 'grad_norm': 6.2519965171813965, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.4}


 20%|██        | 30/150 [02:12<08:17,  4.15s/it]

...

{'eval_loss': 0.004651120863854885, 'eval_runtime': 14.9022, 'eval_samples_per_second': 13.421, 'eval_steps_per_second': 0.872, 'epoch': 3.0}


100%|██████████| 150/150 [14:18<00:00,  3.64s/it]

{'train_runtime': 858.8854, 'train_samples_per_second': 2.794, 'train_steps_per_second': 0.175, 'train_loss': 0.29753789767622946, 'epoch': 3.0}


100%|██████████| 150/150 [14:20<00:00,  5.74s/it]


TrainOutput(global_step=150, training_loss=0.29753789767622946, metrics={'train_runtime': 858.8854, 'train_samples_per_second': 2.794, 'train_steps_per_second': 0.175, 'total_flos': 16033329936000.0, 'train_loss': 0.29753789767622946, 'epoch': 3.0})

In [8]:
# Save trained model
model.save_pretrained("bert_medical_classifier_pytorch")
tokenizer.save_pretrained("bert_medical_classifier_pytorch")

# Load model later
from transformers import BertForSequenceClassification, BertTokenizer

model = BertForSequenceClassification.from_pretrained("bert_medical_classifier_pytorch")
tokenizer = BertTokenizer.from_pretrained("bert_medical_classifier_pytorch")
